# KAN-CDSCO2004U  Machine Learning and Deep Learning

## Lab 6: SVM for Face Recognition
**Estimated time: 1 hour**

### Learning Objectives
By the end of this exercise, you will be able to:
- Understand how **Support Vector Machines (SVM)** work for classification
- Use **PCA** for dimensionality reduction and build a **pipeline** with SVM
- Perform **hyperparameter tuning** using **Grid Search** with cross-validation
- Evaluate a classifier using **classification reports** and **confusion matrices**
- Apply SVM to a real-world **face recognition** problem

In this exercise, you will implement an **SVM-based face recognition** system using techniques from Lecture 6. You will reduce dimensionality with PCA, tune hyperparameters with Grid Search, and evaluate your model's performance.

**How to work through this notebook:**
- 🏃 **RUN** cells = Just execute the code to see the output
- ✏️ **TODO** cells = Write your own code or answer questions
- 📖 **READ** cells = Explanations to help you understand the concepts

---
## Setup

🏃 **RUN** the cell below to import libraries.

In [ ]:
# Import needed libraries
# Author: Luca Gudi (lgg.digi@cbs.dk)

from sklearn.datasets import fetch_lfw_people
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.decomposition import PCA as RandomizedPCA
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import classification_report, confusion_matrix

---
## 1. Import and Explore the Data

📖 **READ**: We will work with the **Labeled Faces in the Wild (LFW)** dataset — a collection of face photographs designed for studying unconstrained face recognition.

Key facts:
- Each image is **62 × 47 pixels** (~3,000 pixel values per image)
- We filter to keep only people with **at least 60 images** in the dataset
- The goal is to **classify whose face** appears in a given image

🏃 **RUN** the cell below to load the data.

In [ ]:
# Load the LFW dataset (only people with at least 60 images)
faces = fetch_lfw_people(min_faces_per_person=60)
print(f"Target names: {faces.target_names}")
print(f"Images shape: {faces.images.shape}")
print(f"Number of samples: {faces.images.shape[0]}")
print(f"Image size: {faces.images.shape[1]} x {faces.images.shape[2]} pixels")

✏️ **TODO**: Plot a grid of faces to see what the data looks like. Display **3 rows × 5 columns** of face images with the person's name as the label.

*Hints:*
- *Use `plt.subplots(3, 5)` to create the grid*
- *Loop over `ax.flat` and use `axi.imshow(faces.images[i], cmap='bone')`*
- *Set the xlabel to `faces.target_names[faces.target[i]]`*
- *Remove ticks with `axi.set(xticks=[], yticks=[])`*

In [ ]:
# TODO: Plot a 3x5 grid of faces with their names
# fig, ax = plt.subplots(...)
# for i, axi in enumerate(ax.flat):
#     ...


---
## 2. Build the Model Pipeline (PCA + SVM)

📖 **READ**: Each image has ~3,000 pixel values — that's a lot of features! We use **Principal Component Analysis (PCA)** to reduce this to **150 components** before feeding the data into the SVM classifier.

Instead of applying PCA and SVM separately, we package them into a **pipeline**:
1. **PCA** extracts the 150 most important features (with whitening)
2. **SVC** (Support Vector Classifier) uses an **RBF kernel** to classify the faces

Using `make_pipeline` ensures that PCA and SVM are always applied together — no data leakage!

✏️ **TODO**: Create a pipeline that combines PCA (150 components) with an SVM classifier (RBF kernel).

*Hints:*
- *Use `RandomizedPCA(n_components=150, whiten=True, random_state=42)` for PCA*
- *Use `SVC(kernel='rbf', class_weight='balanced')` for the classifier*
- *Combine them with `make_pipeline(pca, svc)`*

In [ ]:
# TODO: Create a PCA object with 150 components
# pca = ...

# TODO: Create an SVC object with RBF kernel
# svc = ...

# TODO: Combine them into a pipeline
# model = ...


---
## 3. Train-Test Split

📖 **READ**: Before training, we split the data into **training** and **test** sets. The model will learn patterns from the training set and be evaluated on the unseen test set.

Note: We use `faces.data` (the flattened pixel values) rather than `faces.images` (the 2D arrays), because sklearn models expect 2D input arrays of shape `(n_samples, n_features)`.

✏️ **TODO**: Split the data into training and test sets.

*Hints:*
- *Use `train_test_split(faces.data, faces.target, random_state=42)`*
- *Store the results in `Xtrain, Xtest, ytrain, ytest`*

In [ ]:
# TODO: Split the data into training and test sets
# Xtrain, Xtest, ytrain, ytest = ...


---
## 4. Hyperparameter Tuning with Grid Search

📖 **READ**: SVM performance depends heavily on two hyperparameters:
- **C** (regularization): Controls the trade-off between margin width and classification error
  - Large C → narrow margin, risk of overfitting
  - Small C → wide margin, better generalization
- **gamma** (RBF kernel width): Controls how far the influence of a training example reaches
  - Large gamma → close reach, complex boundary
  - Small gamma → far reach, smoother boundary

**Grid Search** systematically tries all combinations of C and gamma values using **cross-validation** to find the best pair.

✏️ **TODO**: Use `GridSearchCV` to find the best combination of C and gamma.

*Hints:*
- *Define a parameter grid: `{'svc__C': [1, 5, 10, 50], 'svc__gamma': [0.0001, 0.0005, 0.001, 0.005]}`*
- *Note the `svc__` prefix — this tells the grid search to tune the SVC step inside the pipeline*
- *Use `GridSearchCV(model, param_grid)` and call `.fit(Xtrain, ytrain)`*
- *Print `grid.best_params_` to see the winning combination*

In [ ]:
# TODO: Define the parameter grid
# param_grid = ...

# TODO: Create a GridSearchCV object and fit it
# grid = ...
# grid.fit(...)

# TODO: Print the best parameters
# print(grid.best_params_)


### ✏️ TODO: Answer the following questions

**Q1: Why do we use Grid Search instead of manually trying different hyperparameter values? What role does cross-validation play in this process?**

Your answer: ___

**Q2: What do the parameters C and gamma control in an SVM with an RBF kernel? What happens if C is too large or too small?**

Your answer: ___

---
## 5. Prediction and Evaluation

✏️ **TODO**: Use the best model from Grid Search to predict labels for the test data.

*Hints:*
- *Get the best model with `grid.best_estimator_`*
- *Call `.predict(Xtest)` to get predicted labels*

In [ ]:
# TODO: Get the best model and predict on test data
# model = grid.best_estimator_
# yfit = ...


✏️ **TODO**: Visualize some test images with their **predicted names**. Color incorrect predictions in **red**.

*Hints:*
- *Use `plt.subplots(4, 6)` for a 4×6 grid*
- *Reshape each test image back to 62×47 with `Xtest[i].reshape(62, 47)`*
- *Set the ylabel to the predicted name: `faces.target_names[yfit[i]].split()[-1]`*
- *Use `color='black'` if correct, `color='red'` if wrong*

In [ ]:
# TODO: Plot test images with predicted labels
# fig, ax = plt.subplots(4, 6)
# for i, axi in enumerate(ax.flat):
#     ...
# fig.suptitle('Predicted Names; Incorrect Labels in Red', size=14)


✏️ **TODO**: Print the **classification report** to see precision, recall, and F1-score for each person.

*Hints:*
- *Use `classification_report(ytest, yfit, target_names=faces.target_names)`*

In [ ]:
# TODO: Print the classification report
# print(classification_report(...))


✏️ **TODO**: Display the **confusion matrix** as a heatmap to see which faces are being confused with each other.

*Hints:*
- *Use `confusion_matrix(ytest, yfit)` to compute the matrix*
- *Use `sns.heatmap(mat.T, square=True, annot=True, fmt='d', cbar=False)` to plot it*
- *Set `xticklabels` and `yticklabels` to `faces.target_names`*

In [ ]:
# TODO: Compute and display the confusion matrix
# mat = confusion_matrix(...)
# sns.heatmap(...)
# plt.xlabel('true label')
# plt.ylabel('predicted label')


### ✏️ TODO: Answer the following questions

**Q3: Looking at the classification report, which person has the highest precision? Which has the lowest recall? What might explain the difference?**

Your answer: ___

**Q4: Looking at the confusion matrix, which pairs of faces are most commonly confused? Why might the model struggle with those?**

Your answer: ___

---
## Summary

In this lab, you learned how to:

| Section | Technique | Python Code / sklearn Class |
| :--- | :--- | :--- |
| **1. Data Exploration** | Load & visualize face images | `fetch_lfw_people(min_faces_per_person=60)` |
| **2. Model Pipeline** | PCA + SVM in a pipeline | `make_pipeline(PCA(...), SVC(...))` |
| **3. Train-Test Split** | Split data for evaluation | `train_test_split(X, y)` |
| **4. Grid Search** | Hyperparameter tuning with CV | `GridSearchCV(model, param_grid)` |
| **5. Prediction** | Predict & visualize results | `model.predict(Xtest)` |
| **5. Evaluation** | Classification report & confusion matrix | `classification_report(...)`, `confusion_matrix(...)` |

**Key takeaways:**
- **PCA** reduces high-dimensional image data to a manageable number of features
- **SVM with RBF kernel** is powerful for classification, but requires tuning **C** and **gamma**
- **Grid Search with CV** automates hyperparameter tuning and avoids overfitting to a single split
- **Classification reports** and **confusion matrices** reveal per-class strengths and weaknesses
- Always evaluate your model on **unseen test data** to measure true generalization